# Safecast 空間線量率データ フォーマッター

[Safecast API](https://api.safecast.org/en-US/home) から放射線測定データを取得し、
既存のボロノイ可視化で扱える対応幅 CSV に整形する。

**処理の流れ:**

1. **取得**: 福島周辺（緯度・経度を中心とする半径 100km）の測定データを月別に API からダウンロードして `raw/` にキャッシュ
2. **整形**: 緯度経度はそのまま（小数）使い、時刻を **日単位（day）** に丸めて「観測点 × 日」のテーブルに整形
   - 同一の (lat, lon) かつ同一日の測定は平均
   - 別の日の測定は別の列にする
3. **単位変換**: cpm（counts per minute）→ μSv/h（換算係数 350 cpm/μSv/h、Safecast 公式）
4. **出力**: `name, lat, lon, 2011_04_24, 2011_04_25, ...` 形式で `../safecast.csv` に書き出し

**生成後の使い方:**

```bash
# 単日のボロノイ図（任意の日カラム名を指定）
uv run python main.py safecast 2011_04_24

# 時系列動画（GIF / MP4）
uv run python scripts/generate_voronoi_video.py safecast
```

**注意:** Safecast はモバイル計測（自転車・車に bGeigie デバイスを載せて移動しながら計測）が中心のため、
同じ (lat, lon) で複数回観測されることはほぼない。
そのため大半の行は1つの日カラムにのみ値が入る、いわゆる **疎な対応幅 CSV** になる。

In [22]:
import json
import time
from pathlib import Path

import pandas as pd
import requests

# ── パス ────────────────────────────────────────────────
RAW_DIR = Path('raw')
OUT_CSV = Path('../safecast_2023_8.csv')
RAW_DIR.mkdir(exist_ok=True)

# ── Safecast API 設定 ───────────────────────────────────
API_URL = 'https://api.safecast.org/measurements.json'
PER_PAGE = 1000          # 1ページあたり最大件数（API上限）
MAX_PAGES = 200          # 月あたりの最大ページ数（暴走防止）
API_WAIT_SEC = 0.3       # ページ間の待機時間（API負荷軽減）
REQUEST_TIMEOUT = 30     # リクエストタイムアウト（秒）

# ── 取得対象エリア（福島周辺・半径100km）────────────────
CENTER_LAT = 37.5        # 福島県の中央付近
CENTER_LON = 140.5
DISTANCE_M = 100_000     # メートル単位（API仕様）

# ── 取得対象期間 ────────────────────────────────────────
# 形式は ('YYYY_MM', 開始ISO日付, 終了ISO日付(翌月1日))


def _build_monthly_periods() -> list[tuple[str, str, str]]:
    """各月の (期間ラベル, 開始ISO日付, 終了ISO日付) のリストを生成する。"""
    periods: list[tuple[str, str, str]] = []
    months = [
        (2023, 8),
    ]
    for year, month in months:
        label = f'{year}_{month:02d}'
        start = f'{year}-{month:02d}-01T00:00:00Z'
        # 翌月の1日を end とする（exclusive な上限になる）
        next_year = year + (1 if month == 12 else 0)
        next_month = 1 if month == 12 else month + 1
        end = f'{next_year}-{next_month:02d}-01T00:00:00Z'
        periods.append((label, start, end))
    return periods


MONTHLY_PERIODS = _build_monthly_periods()

# ── 単位変換 ────────────────────────────────────────────
# Safecast 公式換算係数: 350 cpm = 1 μSv/h（LND-7317 ガイガー管基準）
CPM_PER_USVH = 350.0

# ── 観測点の同一視 ──────────────────────────────────────
# 緯度経度を浮動小数点のまま比較するとほぼ同じ位置でも別観測点扱いになる。
# そこで小数点以下を一定桁で丸めることで「同一観測点」を判定する。
# 6桁 ≒ 11cm 精度。Safecast の bGeigie のGPS精度を考えれば十分。
COORD_ROUND_DIGITS = 6

print(
    f'対象エリア: 緯度 {CENTER_LAT}, 経度 {CENTER_LON} を中心とする半径 {DISTANCE_M//1000} km')
print(
    f'対象期間: {MONTHLY_PERIODS[0][0]} 〜 {MONTHLY_PERIODS[-1][0]}（{len(MONTHLY_PERIODS)} ヶ月）')
print(
    f'位置の同一視: 小数点以下 {COORD_ROUND_DIGITS} 桁で丸めて同一観測点とみなす')

対象エリア: 緯度 37.5, 経度 140.5 を中心とする半径 100 km
対象期間: 2023_08 〜 2023_08（1 ヶ月）
位置の同一視: 小数点以下 6 桁で丸めて同一観測点とみなす


In [17]:
# ── 1. Safecast API から月別データを取得 ─────────────────
# 各月のデータを raw/<period>.json に JSON Lines として保存する。
# 既存ファイルがあればダウンロードをスキップ（毎回 API を叩かない）。

session = requests.Session()


def fetch_one_page(period_start: str, period_end: str, page: int) -> list[dict]:
    """1ページぶんの測定データを Safecast API から取得する。"""
    params = {
        'latitude': CENTER_LAT,
        'longitude': CENTER_LON,
        'distance': DISTANCE_M,
        'captured_after': period_start,
        'captured_before': period_end,
        'per_page': PER_PAGE,
        'page': page,
    }
    resp = session.get(API_URL, params=params, timeout=REQUEST_TIMEOUT)
    resp.raise_for_status()
    return resp.json()


def fetch_all_for_period(period_label: str, start_iso: str, end_iso: str) -> list[dict]:
    """指定期間の全測定データをページングしながら取得する。"""
    all_records: list[dict] = []
    for page in range(1, MAX_PAGES + 1):
        try:
            records = fetch_one_page(start_iso, end_iso, page)
        except Exception as e:
            print(f'    ✗ ページ {page} 取得失敗: {e}')
            break
        if not records:
            # 空レスポンス = 全件取得完了
            break
        all_records.extend(records)
        print(f'    page {page}: +{len(records)}件 (累計 {len(all_records)})')
        time.sleep(API_WAIT_SEC)
    else:
        # for-else: break しなかったとき（=MAX_PAGES に達した）
        print(f'    ⚠ MAX_PAGES={MAX_PAGES} に到達。データが切り捨てられた可能性あり。')
    return all_records


def download_period(period_label: str, start_iso: str, end_iso: str) -> int:
    """期間データを取得して raw/<period>.json に保存。既存ならスキップ。"""
    dest = RAW_DIR / f'{period_label}.json'
    if dest.exists():
        # キャッシュ件数を返すだけ（再ダウンロードしない）
        with dest.open(encoding='utf-8') as f:
            cached = json.load(f)
        print(f'  {period_label}: スキップ（既存 {len(cached)}件）')
        return len(cached)

    print(f'  {period_label}: 取得開始 ({start_iso} 〜 {end_iso})')
    records = fetch_all_for_period(period_label, start_iso, end_iso)
    with dest.open('w', encoding='utf-8') as f:
        json.dump(records, f, ensure_ascii=False)
    print(f'  ✓ {period_label}: 保存完了 {len(records)}件')
    return len(records)


print('Safecast API からデータ取得開始...\n')
period_counts: dict[str, int] = {}
for period_label, start_iso, end_iso in MONTHLY_PERIODS:
    period_counts[period_label] = download_period(
        period_label, start_iso, end_iso)

print(f'\n=== 取得完了 ===')
total = sum(period_counts.values())
for period_label, count in period_counts.items():
    print(f'  {period_label}: {count:>6,} 件')
print(f'  ----------------')
print(f'  合計      : {total:>6,} 件')

Safecast API からデータ取得開始...

  2023_08: 取得開始 (2023-08-01T00:00:00Z 〜 2023-09-01T00:00:00Z)
    page 1: +1000件 (累計 1000)
    page 2: +1000件 (累計 2000)
    page 3: +1000件 (累計 3000)
    page 4: +1000件 (累計 4000)
    page 5: +1000件 (累計 5000)
    page 6: +1000件 (累計 6000)
    page 7: +1000件 (累計 7000)
    page 8: +1000件 (累計 8000)
    page 9: +1000件 (累計 9000)
    page 10: +1000件 (累計 10000)
    page 11: +1000件 (累計 11000)
    page 12: +1000件 (累計 12000)
    page 13: +1000件 (累計 13000)
    page 14: +1000件 (累計 14000)
    page 15: +1000件 (累計 15000)
    page 16: +1000件 (累計 16000)
    page 17: +1000件 (累計 17000)
    page 18: +1000件 (累計 18000)
    page 19: +1000件 (累計 19000)
    page 20: +1000件 (累計 20000)
    page 21: +1000件 (累計 21000)
    page 22: +1000件 (累計 22000)
    page 23: +1000件 (累計 23000)
    page 24: +1000件 (累計 24000)
    page 25: +1000件 (累計 25000)
    page 26: +1000件 (累計 26000)
    page 27: +1000件 (累計 27000)
    page 28: +1000件 (累計 28000)
    page 29: +1000件 (累計 29000)
    page 30: +1000件 (累計 30000)

In [23]:
# ── 2. レコードを「日単位 × 精密 (lat, lon)」に集計 ────────────────
# 各レコードを以下のように整形する:
#   - (lat, lon): API のものをそのまま（COORD_ROUND_DIGITS 桁で丸める）
#   - 時刻: captured_at を日単位に切り捨て 'YYYY_MM_DD' 形式の文字列に
#   - 値: cpm を μSv/h に換算（÷350）
#
# 同一 (lat, lon, 日) に複数測定があった場合は平均する。
# unit が 'cpm' 以外、または value <= 0 のレコードは除外する。

def load_period_records(period_label: str) -> list[dict]:
    """raw/<period>.json を読み込んで生レコードのリストを返す。"""
    path = RAW_DIR / f'{period_label}.json'
    if not path.exists():
        return []
    with path.open(encoding='utf-8') as f:
        return json.load(f)


def records_to_dataframe(records: list[dict]) -> pd.DataFrame:
    """生レコード（dict のリスト）をクリーニング済み DataFrame に変換する。

    - cpm 以外、value <= 0、緯度経度欠損は除外
    - lat / lon を COORD_ROUND_DIGITS 桁で丸める（同一点判定のため）
    - captured_at を日単位の 'YYYY_MM_DD' 文字列に変換（時系列カラム名）
    - value を μSv/h に換算
    """
    if not records:
        return pd.DataFrame(columns=['lat', 'lon', 'day_col', 'value_usvh'])

    df = pd.DataFrame(records)
    # cpm のみ
    df = df[df['unit'] == 'cpm']
    # 異常値除外
    df = df[df['value'] > 0]
    # 緯度経度欠損除外
    df = df.dropna(subset=['latitude', 'longitude', 'captured_at'])
    if df.empty:
        return pd.DataFrame(columns=['lat', 'lon', 'day_col', 'value_usvh'])

    # 緯度経度を丸める
    df['lat'] = df['latitude'].round(COORD_ROUND_DIGITS)
    df['lon'] = df['longitude'].round(COORD_ROUND_DIGITS)

    # 時刻を日単位に切り捨て → 'YYYY_MM_DD' 形式に
    ts = pd.to_datetime(df['captured_at'], utc=True).dt.floor('D')
    df['day_col'] = ts.dt.strftime('%Y_%m_%d')

    # cpm → μSv/h
    df['value_usvh'] = df['value'] / CPM_PER_USVH

    return df[['lat', 'lon', 'day_col', 'value_usvh']]


# ── 全月のレコードを連結 ─────────────────────────────────────
print('生データを読み込んで整形中...')
all_dfs: list[pd.DataFrame] = []
for period_label, _, _ in MONTHLY_PERIODS:
    records = load_period_records(period_label)
    df_period = records_to_dataframe(records)
    print(
        f'  {period_label}: 生 {len(records):>6,} 件 → 有効 {len(df_period):>6,} 件')
    if not df_period.empty:
        all_dfs.append(df_period)

if not all_dfs:
    raise RuntimeError(
        '有効なレコードが1件もありません。raw/ にデータがあるか確認してください。')

records_df = pd.concat(all_dfs, ignore_index=True)
print(f'\n合計有効レコード: {len(records_df):,} 件')

# ── (lat, lon, day_col) で平均 ────────────────────────────
# 同一観測点・同一日に複数測定があった場合は平均する
agg_df = (
    records_df
    .groupby(['lat', 'lon', 'day_col'], as_index=False)['value_usvh']
    .mean()
)
print(f'集計後（lat × lon × 日 のユニーク組合せ）: {len(agg_df):,} 行')

# ユニーク観測点数 / ユニーク日数の確認
n_points = agg_df.groupby(['lat', 'lon']).ngroups
n_days = agg_df['day_col'].nunique()
print(f'  ユニーク観測点数: {n_points:,}')
print(f'  ユニーク日数    : {n_days:,}')

生データを読み込んで整形中...
  2023_08: 生 200,000 件 → 有効 128,458 件

合計有効レコード: 128,458 件
集計後（lat × lon × 日 のユニーク組合せ）: 10,048 行
  ユニーク観測点数: 9,641
  ユニーク日数    : 20


In [24]:
# ── 3. ピボットして対応幅 CSV を出力 ──────────────────────────
# 形式:
#   name, lat, lon, 2011_04_24, 2011_04_25, ...
#
# 行   : 各観測点（unique な lat, lon の組み合わせ）
# 列   : 日タイムスタンプ（YYYY_MM_DD）
# 値   : その観測点・その日の μSv/h（無ければ NaN）

# pivot: index=(lat, lon), columns=day_col, values=value_usvh
wide_df = (
    agg_df.pivot_table(
        index=['lat', 'lon'],
        columns='day_col',
        values='value_usvh',
        aggfunc='mean',
    )
    .reset_index()
)

# 日カラムを日付順にソートする（pivot 直後だと辞書順だが念のため）
day_cols = sorted([c for c in wide_df.columns if c not in ('lat', 'lon')])
wide_df = wide_df[['lat', 'lon'] + day_cols]

# name 列を先頭に追加（点のラベル。緯度経度から一意なID を生成）
wide_df.insert(
    0,
    'name',
    [
        f'p_{lat:.6f}_{lon:.6f}'
        for lat, lon in zip(wide_df['lat'], wide_df['lon'])
    ],
)

OUT_CSV.parent.mkdir(parents=True, exist_ok=True)
wide_df.to_csv(OUT_CSV, index=False, encoding='utf-8')

print(f'出力完了: {OUT_CSV.resolve()}')
print(f'観測点数: {len(wide_df):,}')
print(f'日カラム数: {len(day_cols):,}')
print(f'最初の日カラム: {day_cols[0]}')
print(f'最後の日カラム: {day_cols[-1]}\n')

# サンプル: 値の高い順に 10 行表示
print('--- 値の高い上位 10 観測点（最大 μSv/h ベース）---')
max_per_point = wide_df[day_cols].max(axis=1)
top10_idx = max_per_point.sort_values(ascending=False).head(10).index
print(
    wide_df.loc[top10_idx, ['name', 'lat', 'lon']]
    .assign(max_uSv_h=max_per_point.loc[top10_idx].round(3).values)
    .to_string(index=False)
)

print('\n--- 全期間サマリ ---')
all_values = wide_df[day_cols].stack()
print(all_values.describe().round(3))

出力完了: /Users/rosalina_0106/Develop/ginder/data/safecast/safecast_2023_8.csv
観測点数: 9,641
日カラム数: 20
最初の日カラム: 2023_08_01
最後の日カラム: 2023_08_20

--- 値の高い上位 10 観測点（最大 μSv/h ベース）---
                  name       lat        lon  max_uSv_h
p_37.417278_141.009369 37.417278 141.009369      4.374
p_37.417290_141.009460 37.417290 141.009460      4.357
p_37.417110_141.009033 37.417110 141.009033      4.339
p_37.417114_141.009293 37.417114 141.009293      4.200
p_37.407742_141.015442 37.407742 141.015442      1.663
p_37.407665_141.015350 37.407665 141.015350      1.539
p_37.407803_141.015518 37.407803 141.015518      1.534
p_37.407642_141.015228 37.407642 141.015228      1.533
p_37.407730_141.015305 37.407730 141.015305      1.504
p_37.407787_141.015182 37.407787 141.015182      1.499

--- 全期間サマリ ---
count    10048.000
mean         0.134
std          0.258
min          0.031
25%          0.091
50%          0.106
75%          0.123
max          4.374
dtype: float64
